In [194]:
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import statsmodels.api as sm
# Direct Plotting
from statsmodels.graphics.tsaplots import plot_acf
# adfuller and other plots
from statsmodels.tsa.stattools import adfuller, acf, pacf
# VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
# Smf without assining X
import statsmodels.formula.api as smf

from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")

In [195]:
end = dt.date.today()
start = end - dt.timedelta(days=365*10)
df = yf.download('GOOGL', start = start, end = end, interval = '1d',auto_adjust=False,multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


# Rolling Multiple Linear Regression

In [196]:
df['return'] = df['Adj Close'].pct_change()
df['lag1'] = df['return'].shift(1)
df['lag2'] = df['return'].shift(2)
df['lag3'] = df['return'].shift(3)
df = df.dropna()
X = df[['lag1','lag2','lag3']]
y = df['return']

window = 50
preds = []
coeff = []

for i in range (window, len(X)):
    X_train = X[i-window:i]
    y_train = y[i-window:i]

    X_test = X[i:i+1]
    model = LinearRegression()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    preds.append(pred[0])
    coeff.append(model.coef_)

y_actual = y[window:window + len(preds)]
corr = np.corrcoef(preds, y_actual)[0,1]
print("Correlation:", corr)



Correlation: 0.06626952247239908


# Ridge Regression

In [197]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100]

In [198]:
from sklearn.linear_model import Ridge
import numpy as np

alphas = [0.001, 0.01, 0.1, 1, 10, 100]

for a in alphas:

    preds = []

    for i in range(window, len(X)):

        X_train = X[i-window:i]
        y_train = y[i-window:i]

        X_test = X[i:i+1]

        model = Ridge(alpha=a)
        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        preds.append(pred[0])

    y_actual = y[window:window + len(preds)]

    corr = np.corrcoef(preds, y_actual)[0,1]

    print("alpha:", a, "corr:", corr)

alpha: 0.001 corr: 0.06319614841252544
alpha: 0.01 corr: 0.05338685500865441
alpha: 0.1 corr: 0.019682621838935137
alpha: 1 corr: -0.016716687016259734
alpha: 10 corr: -0.023249158443198234
alpha: 100 corr: -0.023941884981037997


# Adding More Features

In [199]:
end = dt.date.today()
start = end - dt.timedelta(days=365*10)
df = yf.download('GOOGL', start = start, end = end, interval = '1d',auto_adjust=False,multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [200]:
df['return'] = df['Adj Close'].pct_change()
df['momentum'] = df['Adj Close'].pct_change(5).shift(1)

df['obv'] = (np.sign(df['return']) * df['Volume']).cumsum()
df['obv_diff'] = df['obv'].diff().shift(1)

df['volatility'] = df['return'].rolling(10).std()
df.dropna(inplace=True)

In [201]:
X = df[['obv_diff','volatility','momentum']]
y = df['return']

X = X.reset_index(drop=True)
y = y.reset_index(drop=True)
alphas = [0.001, 0.01, 0.1, 1, 10, 100]

for a in alphas:

    preds = []

    for i in range(window, len(X)):

        X_train = X[i-window:i]
        y_train = y[i-window:i]

        X_test = X[i:i+1]

        model = Ridge(alpha=a)
        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        preds.append(pred[0])

    y_actual = y[window:window + len(preds)]

    corr = np.corrcoef(preds, y_actual)[0,1]

    print("alpha:", a, "corr:", corr)

alpha: 0.001 corr: 0.03259508311729953
alpha: 0.01 corr: 0.035750003449286534
alpha: 0.1 corr: 0.040481817334080496
alpha: 1 corr: 0.03996134153915024
alpha: 10 corr: 0.039400495514764176
alpha: 100 corr: 0.03932960446612318


# Lasso Regression

In [202]:
end = dt.date.today()
start = end - dt.timedelta(days=365*10)
df1 = yf.download('GOOGL', start = start, end = end, interval = '1d',auto_adjust=False,multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [203]:
df1['return'] = df1['Adj Close'].pct_change()
for i in range(1, 6):
    df1[f'lag{i}'] = df1['return'].shift(i)

In [204]:
df1['mom3'] = df1['Adj Close'].pct_change(3).shift(1)
df1['mom5'] = df1['Adj Close'].pct_change(5).shift(1)
df1['mom10'] = df1['Adj Close'].pct_change(10).shift(1)

In [205]:
df1['mr10'] = (df1['Adj Close'] - df1['Adj Close'].rolling(10).mean()).shift(1)
df1['mr20'] = (df1['Adj Close'] - df1['Adj Close'].rolling(20).mean()).shift(1)

In [206]:
df1['vol5'] = df1['return'].rolling(5).std().shift(1)
df1['vol10'] = df1['return'].rolling(10).std().shift(1)
df1['vol20'] = df1['return'].rolling(20).std().shift(1)

In [207]:
df1['obv_diff'] = (np.sign(df1['return']) * df1['Volume']).shift(1)
df1 = df1.dropna()

In [208]:
X = df1[[
    'lag1','lag2','lag3','lag4','lag5',
    'mom3','mom5','mom10',
    'mr10','mr20',
    'vol5','vol10','vol20',
    'obv_diff'
]]

y = df1['return']

X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

In [213]:
from sklearn.linear_model import Lasso
alphas = [0.00001, 0.0001, 0.001]
for a in alphas:
    preds = []
    for i in range(window, len(X)):
        X_train = X[i-window:i]
        y_train = y[i-window:i]
        X_test = X[i:i+1]
        model = Lasso(alpha=a)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        preds.append(pred[0])

    y_actual = y[window:window + len(preds)]
    corr = np.corrcoef(preds, y_actual)[0,1]
print("alpha:", a)
print("corr:", corr)
print("coeff:", model.coef_)

alpha: 0.001
corr: 0.023311437057335654
coeff: [ 0.00000000e+00  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00  0.00000000e+00  0.00000000e+00 -0.00000000e+00
  2.07742457e-04 -2.77055209e-04  0.00000000e+00  0.00000000e+00
  0.00000000e+00 -2.60303573e-11]


# Elastic Net

In [215]:
from sklearn.linear_model import ElasticNet
import numpy as np

l1_ratios = [0.2, 0.5, 0.8]

for l1 in l1_ratios:

    preds = []

    for i in range(window, len(X)):

        X_train = X[i-window:i]
        y_train = y[i-window:i]

        X_test = X[i:i+1]

        model = ElasticNet(alpha=0.001, l1_ratio=l1)

        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        preds.append(pred[0])

    y_actual = y[window:window + len(preds)]

    corr = np.corrcoef(preds, y_actual)[0,1]

    print("l1_ratio:", l1)
    print("corr:", corr)
    print("coeff:", model.coef_)
    print("------")

l1_ratio: 0.2
corr: 0.017093189248271457
coeff: [ 0.00000000e+00  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00  0.00000000e+00  0.00000000e+00 -0.00000000e+00
  2.85413078e-04 -3.34194982e-04  0.00000000e+00  0.00000000e+00
 -0.00000000e+00 -2.83878946e-11]
------
l1_ratio: 0.5
corr: 0.02002502550326746
coeff: [ 0.00000000e+00  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00  0.00000000e+00  0.00000000e+00 -0.00000000e+00
  2.56271425e-04 -3.12758588e-04  0.00000000e+00  0.00000000e+00
  0.00000000e+00 -2.75031045e-11]
------
l1_ratio: 0.8
corr: 0.022054682568165156
coeff: [ 0.00000000e+00  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00  0.00000000e+00  0.00000000e+00 -0.00000000e+00
  2.27154617e-04 -2.91336963e-04  0.00000000e+00  0.00000000e+00
  0.00000000e+00 -2.66194758e-11]
------
